##### Import libraries

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import spacy
from spacy.matcher import PhraseMatcher

pd.set_option("display.max_columns", None)   # no column truncation
pd.set_option("display.max_rows", None)      # no row truncation
pd.set_option("display.width", None)         # no line-wrapping
pd.set_option("display.max_colwidth", None)

##### First try with Spacy PhraseMatcher

In [2]:
# 1) Minimal ACE lexicon : Need to expand/refine taking into account "terminos.docx" file
# Dict keys: concept_id -> mapped from "terminos.docx" file
# Values: list of terms (lowercase), to be refined 
# Probably need to treat the accentuations.        
ACE_TERMS = {
    "ACE_SEXUAL_ABUSE": [
        "abuso sexual", "abusos sexuales", "violación", "agresión sexual", "tocamientos", "abuso sexual infantil",
        "maltrato sexual", "sexo oral", "penetración", "tocamientos", "voyeurismo", "exhibicionismo", "explotación",
        "explotación sexual", "víctima de agresión sexual"
    ],
    "ACE_PHYSICAL_ABUSE": [
        "maltrato físico", "golpes", "agresión física", "palizas", "violencia física",
        "malos tratos físicos", "abuso físico", "golpeado", "patada", "puñetazo", "azotes", 
        "agitar", "empujar", "arañar", "pellizcar", "morder", "estrangular", "torturar", "quemar", 
        "lanzar", "sacudir", "síndrome del bebé zarandeado", "síndrome de Munchausen",
        "víctima de abuso físico"
    ],
    "ACE_EMOTIONAL_ABUSE": [
        "maltrato psicológico", "abuso emocional", "humillaciones", "insultos", "amenazas",
        "abuso psicológico", "malos tratos psicológicos", "maltrato emocional", "maltrato verbal",
        "víctima de abuso emocional", "víctima de abuso psicológico"
    ],
    "ACE_PHYSICAL_NEGLECT": [
        "abandono", "falta de cuidados",  "descuido"
    ],
    "ACE_EMOTIONAL_NEGLECT": [
        "desatención", # Refine with appropiate terms
    ],
    "ACE_UNSPECIFIED_NEGLECT": [
        "negligencia", "comportamiento negligente", "falta de cuidados",  
    ],
    "ACE_DOMESTIC_VIOLENCE_EXPOSURE": [
        "violencia intrafamiliar", "violencia doméstica", "violencia en casa", "malos tratos en el hogar",
        "testigo de violencia doméstica"
    ],
    "ACE_PARENT_SUBSTANCE": [
        "padre alcohólico", "madre alcohólica", "consumo de alcohol del padre", "drogadicción del padre",
        "consumo de drogas en casa"
    ],
    "ACE_PARENT_MENTAL_ILLNESS": [
        "madre con depresión", "padre con depresión", "trastorno mental del padre", "psicosis del padre"
    ],
    "ACE_PARENT_INCARCERATION": [
        "padre en prisión", "madre en prisión", "encarcelamiento del padre"
    ],
    "ACE_PARENT_DIVORCE_SEPARATION": [
        "divorcio de los padres", "separación de los padres", "padres separados", "padres divorciados"
    ],
    "ACE_POVERTY_UNEMPLOYMENT": [
        "padres desempleados", "padres en paro", "padres sin empleo", 
        "padre desempleado", "padre en paro", "padre sin empleo", "padre no trabaja",
        "madre desempleada", "madre en paro", "madre sin empleo", "madre no trabaja", 
        "sin medios", "medios limitados", "medios económicos limitados" # Refine with appropiated terms
    ],
    "ACE_GENERAL_TRAUMA": [
        "adversidad precoz", "trauma precoz", "traumatismo precoz", "evento traumático", "evento adverso",
        "situación traumática", "vivencia traumática", "antecedente traumático", 
        "historia de trauma", "experiencia traumática", "suceso traumático",
        "violencia", "violento", "testigo de violencia", "víctima" # Refine with appropiated terms
    ], 
}

In [4]:

# Lightweight negation cues. To be improved.
NEG_CUES = re.compile(r"\b(niega|niega[n]?|sin|no|descarta|descartan)\b", re.I)

NEG_CUES = re.compile(
    r"\b("
    r"niega(?:n)?|"
    r"sin|"
    r"descarta(?:n)?|"
    r"no\s+(?:refiere|presenta|consta|evidencia|hay|ha|han|tuvo|tiene|presentó|refirió)"
    r")\b",
    re.I
)

# Temporal cues
import re

CHILD_CUES = re.compile(
    r"\b("
    # life stages
    r"(?:en|desde|durante)\s+(?:la|su)\s+(?:infancia|niñez|adolescencia)|"
    # "when (s)he was a child/minor"
    r"cuando\s+era\s+(?:niño|niña|menor|adolescente)|"
    # explicit "minor"
    r"menor\s+de\s+edad|"
    # explicit age 0-17: "a los 7 años", "con 15 años"
    r"(?:a\s+los|con)\s+(?:[0-9]|1[0-7])\s+a[nñ]os"
    r")\b",
    re.I
)

RECENT_CUES = re.compile(
    r"\b("
    # relative durations
    r"(?:desde\s+hace|hace)\s+\d+\s+(?:a[nñ]os|mes(?:es)?|semanas|d[ií]as|horas)|"
    # "last N days/weeks/months/years"
    r"en\s+los\s+[uú]ltimos\s+\d+\s+(?:d[ií]as|semanas|mes(?:es)?|a[nñ]os)|"
    # adverbs
    r"recientemente|actualmente|en\s+la\s+actualidad|"
    # "last week/month/year", "this week/month/year"
    r"(?:la|este|esta)\s+(?:semana|mes|a[nñ]o)"
    r")\b",
    re.I
)




In [5]:
def temporal_flags_sentence(span):
    sent = span.sent.text
    rel_start = span.start_char - span.sent.start_char
    rel_end   = span.end_char   - span.sent.start_char
    ctx = sent[:rel_start] + " " + sent[rel_end:]

    childhood = bool(CHILD_CUES.search(ctx))
    recent    = bool(RECENT_CUES.search(ctx))
    return childhood, recent

In [6]:
# Second version.
nlp = spacy.load("es_core_news_md", disable=["ner", "parser", "tagger"])
nlp.add_pipe("sentencizer")

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

# Build matcher patterns with a stable "concept_id" (=category here; you can use SNOMED codes instead)
for concept_id, terms in ACE_TERMS.items():
    matcher.add(concept_id, [nlp.make_doc(t) for t in terms])

def temporal_flags_sentence(span):
    sent = span.sent.text
    rel_start = span.start_char - span.sent.start_char
    rel_end   = span.end_char   - span.sent.start_char
    ctx = sent[:rel_start] + " " + sent[rel_end:]

    childhood = bool(CHILD_CUES.search(ctx))
    recent    = bool(RECENT_CUES.search(ctx))
    return childhood, recent

def annotate_ace(text: str):
    doc = nlp(text)
    matches = matcher(doc)

    ann = []
    for match_id, start, end in matches:
        span = doc[start:end]

        # Sentence-bounded context (left/right of the matched span)
        sent = span.sent.text
        rel_start = span.start_char - span.sent.start_char
        rel_end   = span.end_char   - span.sent.start_char
        sent_left  = sent[:rel_start]
        sent_right = sent[rel_end:]

        # Negation cues: check both sides (often left is enough, but this is safer)
        negated = bool(NEG_CUES.search(sent_left) or NEG_CUES.search(sent_right))

        # Temporal cues split into childhood vs recent
        childhood, recent = temporal_flags_sentence(span)

        ann.append({
            "concept_id": nlp.vocab.strings[match_id],  # category or SNOMED code
            "span": span.text,
            "start_char": span.start_char,
            "end_char": span.end_char,
            "negated": negated,
            "childhood_cue": childhood,
            "recent_cue": recent,
        })

    # Deduplicate exact spans (keep first)
    ann = sorted(ann, key=lambda x: (x["start_char"], -x["end_char"]))
    out, seen = [], set()
    for a in ann:
        key = (a["concept_id"], a["start_char"], a["end_char"])
        if key not in seen:
            out.append(a)
            seen.add(key)

    return out

In [7]:
# Example
if __name__ == "__main__":
    note = "Niega maltrato físico. Refiere abuso sexual en la infancia y violencia doméstica en casa."
    print(annotate_ace(note))

[{'concept_id': 'ACE_PHYSICAL_ABUSE', 'span': 'maltrato físico', 'start_char': 6, 'end_char': 21, 'negated': True, 'childhood_cue': False, 'recent_cue': False}, {'concept_id': 'ACE_SEXUAL_ABUSE', 'span': 'abuso sexual', 'start_char': 31, 'end_char': 43, 'negated': False, 'childhood_cue': True, 'recent_cue': False}, {'concept_id': 'ACE_DOMESTIC_VIOLENCE_EXPOSURE', 'span': 'violencia doméstica', 'start_char': 61, 'end_char': 80, 'negated': False, 'childhood_cue': True, 'recent_cue': False}, {'concept_id': 'ACE_GENERAL_TRAUMA', 'span': 'violencia', 'start_char': 61, 'end_char': 70, 'negated': False, 'childhood_cue': True, 'recent_cue': False}]


##### Load abusos file

In [8]:
# Recover the path
directory_path = os.getcwd()
root_path = os.path.dirname(directory_path)
print(root_path)

c:\Users\adeli\Documents 4-Q2\Practicas-ACE


In [9]:
# Try with patients in the file adversidad_precoz.xlsx (confirmed cases)
ace_cases_filepath = os.path.join(root_path, 'Documentación original', "adversidad_precoz.xlsx")
ace_cases_df = pd.read_excel(ace_cases_filepath, na_values=["#NULL!"])

In [ ]:
ace_cases_df.head(1)

In [11]:
old_cols_ace = [
    'ANONIMIZED_TEMPORAL', 'MC', 'MotivodeConsulta', 'Episodio',
       'FechaEpisodio', 'Sexo', 'FechaNacimiento', 'FechaToma',
       'AntecedentesPsiquiátricos', 'Antecedentes', 'ImpresionesSubjetivas',
       'Grado', 'Planificaciónsuicidareciente', 'Grado_A',
       'EJEIDiagnósticoCodificadoCIE', 'EJEIDiagnóstico',
       'Gradodedañomédicocomoresultadodelintentoactual',
       'Historiafamiliardeintentosdesuicidio',
       'EJEIITranstornoPersonalidadyComportamientoL',
       'EJEIITranstornoPersonalidadyComportamiento', 'Ideaciónsuicidareciente',
       'EJEIIITranstornoSomáticoCod1', 'EJEIIITranstornoSomáticoTxt',
       'Númerodeintentosdesuicidioprevios',
       'EJEIVProblemasPsicosocialesyambientales', 'TRATAMIENTOALALTA',
       'EXPLORACIÓNPSICOPATOLÓGICA', 'Ultimo_Episodio', 'Num_Episodios',
       'abuso', 'maltrato', 'filter_$']

new_cols_ace = [
    "Anonymized_Temporal", "RC", "Reason_for_Consultation", "Episode",
    "Episode_Date", "Sex", "DoB", "Assessment_Date",
    "Psychiatric_History", "Medical_History", "Subjective_Impressions",
    "Grade", "Recent_Suicidal_Planning", "Grade_A",
    "EJEI_Coded_Diagnostic_CIE", "EJEI_Diagnostic",
    "Medical_Damage_Severity_From_Current_Attempt",
    "Family_History_of_Suicide_Attempts",
    "EJEI_Personality_and_Behavior_Disorder_L",
    "EJEI_Personality_and_Behavior_Disorder", "Recent_Suicidal_Ideation",
    "EJEI_Somatic_Disorder_Cod1", "EJEI_Somatic_Disorder_Text",
    "Number_of_Previous_Suicide_Attempts",
    "EJEI_Psychosocial_and_Environmental_Issues", "Discharge_Treatment",
    "Psychopathological_Examination", "Last_Episode", "Num_Episodes",
    "Abuse", "Maltreatment", "Filter"]


In [12]:
ace_cases_df = ace_cases_df.rename(columns=dict(zip(old_cols_ace, new_cols_ace)))

In [ ]:
ace_cases_df.head(1)

In [14]:
# Ensure date format
ace_cases_df["Episode_Date"] = pd.to_datetime(ace_cases_df["Episode_Date"], errors="raise", format="%Y-%m-%d %H:%M:%S")
ace_cases_df["Assessment_Date"] = pd.to_datetime(ace_cases_df["Assessment_Date"], errors="raise", format="%Y-%m-%d %H:%M:%S")
ace_cases_df["DoB"] =  pd.to_datetime(ace_cases_df["DoB"], errors="raise", format="%Y-%m-%d")

In [15]:
# Select a record
test_case = ace_cases_df.iloc[0]
note_ace = test_case["Psychiatric_History"]

In [16]:
note_ace

'Primer contacto con psiquiatría en el 2008, cuando tuvo varios ingresos por intoxicaciones etilicas y abuso de alcohol que relaciona con haber sido victima de malos tratos por parte del padre de su hijo, y abusos sexuales en la infancia.\nRealizó desentoxicacion en el Centro Los Almendros, por un año. En el 2012 empezó seguimiento en San Martin de Valdeiglesias. \nEstuvo ingresada en UDA en Noviembre 2016, 1 mes.\nUn ingreso psiquiátrico en el HRJC en agosto de 2016.\n\nTóxicos:- Alcohol:consumo perjudicial sobre los 25 años, de tipo dipsomaniaco. Niega periodos de consumo diario. Incremento de consumo en el 2008, en contexto de maltratos. Luego abandonó y se ha mantenido abstinente. Refiere en la actualidad consumo ocasional.\n- Cocaína y cannabis: niega consumo a diario pero reconoce consumo varias veces por semana\n- No otros tóxicos.\n\nSB: Natural de España. Madre alcoholica, desde pequeña se quedaba con sus vecinos, que luego acabaron solicitando la tutela. Vivió con sus padres 

In [17]:
annotate_ace(note_ace)

[{'concept_id': 'ACE_SEXUAL_ABUSE',
  'span': 'abusos sexuales',
  'start_char': 206,
  'end_char': 221,
  'negated': False,
  'childhood_cue': True,
  'recent_cue': False},
 {'concept_id': 'ACE_UNSPECIFIED_NEGLECT',
  'span': 'negligencia',
  'start_char': 1102,
  'end_char': 1113,
  'negated': False,
  'childhood_cue': False,
  'recent_cue': False}]

In [ ]:
# https://github.com/xy-leong/anotacionACE.git